<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z339_ExploratorioProdCliente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratorio — Serie de tiempo producto × cliente

Miramos la serie de un producto específico para un cliente específico.
Cambiá `PARAM` para explorar distintas combinaciones.

Modelos que compara:
- OLS (reg lineal clásica)
- Huber (robusta a outliers — los picos pesan menos)
- Lasso (penaliza coeficientes grandes)
- Mediana reciente

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.linear_model import LinearRegression, HuberRegressor, Lasso
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── PARAM ─────────────────────────────────────────────────────────────────────
PARAM = {
    # None → autoselecciona el menos importante
    # int  → usá ese ID directamente
    'product_id': None,
    'customer_id': None,

    # si None, muestra ranking para que elijas
    'top_n_ranking': 10,   # cuántos mostrar en el ranking

    'ventana':    6,        # meses para ajustar los modelos
    'horizonte':  2,        # pasos adelante a predecir
    'huber_eps':  1.35,     # robustez Huber (menor = más robusto)
    'lasso_alpha': 0.1,     # regularización Lasso
}

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
print("Columnas:", dataset.columns)
print(f"Filas: {dataset.height:,}")
dataset.head(5)

# Ranking — elegí producto y cliente

In [ ]:
# columna de cliente — puede llamarse customer_id, customer_code, cod_cliente, etc.
# ajustá si el nombre es distinto
COL_CLIENTE  = 'customer_id'   # ← cambiá si es necesario
COL_PRODUCTO = 'product_id'
COL_TN       = 'tn'
COL_PERIODO  = 'periodo'

# ranking de productos menos importantes (menor volumen total)
rank_prod = (
    dataset.group_by(COL_PRODUCTO)
    .agg(pl.col(COL_TN).sum().alias('tn_total'))
    .sort('tn_total')
)

# ranking de clientes menos importantes
rank_cli = (
    dataset.group_by(COL_CLIENTE)
    .agg(pl.col(COL_TN).sum().alias('tn_total'))
    .sort('tn_total')
)

n = PARAM['top_n_ranking']
print(f"Top {n} productos menos importantes (menor volumen):")
print(rank_prod.head(n))
print()
print(f"Top {n} clientes menos importantes (menor volumen):")
print(rank_cli.head(n))

In [ ]:
# selección automática si no se especificó en PARAM
pid = PARAM['product_id'] if PARAM['product_id'] is not None \
      else int(rank_prod[COL_PRODUCTO][0])

cid = PARAM['customer_id'] if PARAM['customer_id'] is not None \
      else int(rank_cli[COL_CLIENTE][0])

print(f"Producto seleccionado : {pid}")
print(f"Cliente seleccionado  : {cid}")

# serie producto × cliente
serie_df = (
    dataset
    .filter((pl.col(COL_PRODUCTO) == pid) & (pl.col(COL_CLIENTE) == cid))
    .group_by(COL_PERIODO)
    .agg(pl.col(COL_TN).sum())
    .sort(COL_PERIODO)
)

# serie producto agregado (todos los clientes)
serie_prod_df = (
    dataset
    .filter(pl.col(COL_PRODUCTO) == pid)
    .group_by(COL_PERIODO)
    .agg(pl.col(COL_TN).sum())
    .sort(COL_PERIODO)
)

print(f"\nPeríodos con datos producto×cliente: {serie_df.height}")
print(f"Períodos con datos producto total:   {serie_prod_df.height}")
serie_df

# Serie de tiempo — visualización completa

In [ ]:
periodos = serie_df[COL_PERIODO].to_list()
tn       = serie_df[COL_TN].to_numpy().astype(float)
t_idx    = np.arange(len(tn))

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# ── panel 1: serie completa ──
ax = axes[0]
ax.plot(t_idx, tn, 'o-', color='steelblue', markersize=4, linewidth=1.5)
ax.fill_between(t_idx, tn, alpha=0.15, color='steelblue')
ax.set_xticks(t_idx[::3])
ax.set_xticklabels([str(p) for p in periodos[::3]], rotation=45, fontsize=7)
ax.set_title(f'Serie completa — producto {pid} × cliente {cid}', fontsize=10)
ax.set_ylabel('tn')
ax.grid(alpha=0.3)

# ── panel 2: producto total vs cliente ──
ax = axes[1]
periodos_prod = serie_prod_df[COL_PERIODO].to_list()
tn_prod       = serie_prod_df[COL_TN].to_numpy().astype(float)
t_prod        = np.arange(len(tn_prod))

ax.plot(t_prod, tn_prod, 'o-', color='gray', markersize=3,
        linewidth=1.2, alpha=0.6, label=f'producto {pid} total')

# alinear índices
p_set  = set(periodos)
pp_set = set(periodos_prod)
comunes = sorted(p_set & pp_set)
tn_cli_c  = [float(serie_df.filter(pl.col(COL_PERIODO)==p)[COL_TN][0]) for p in comunes]
tn_prod_c = [float(serie_prod_df.filter(pl.col(COL_PERIODO)==p)[COL_TN][0]) for p in comunes]
t_c = np.arange(len(comunes))

ax.plot(t_c, tn_cli_c, 'o-', color='tomato', markersize=4,
        linewidth=1.5, label=f'cliente {cid}')
ax.set_xticks(t_c[::3])
ax.set_xticklabels([str(p) for p in comunes[::3]], rotation=45, fontsize=7)
ax.set_title('Producto total vs cliente seleccionado', fontsize=10)
ax.set_ylabel('tn')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── panel 3: últimos ventana meses + modelos ──
v  = PARAM['ventana']
hz = PARAM['horizonte']
w  = min(v, len(tn))
y  = tn[-w:]
x  = np.arange(w).reshape(-1, 1)
t_ult = t_idx[-w:]

ax = axes[2]
ax.plot(t_ult, y, 'o-', color='steelblue', markersize=5,
        linewidth=2, label='historia reciente')

t_pred = t_ult[-1] + hz

# OLS
ols = LinearRegression().fit(x, y)
p_ols = max(float(ols.predict([[w - 1 + hz]])[0]), 0.0)
x_ext = np.array([t_ult[0], t_pred])
ax.plot(x_ext, ols.predict([[0], [w - 1 + hz]]).flatten(),
        '--', color='tomato', linewidth=1.5, label=f'OLS={p_ols:.2f}')

# Huber
try:
    hub = HuberRegressor(epsilon=PARAM['huber_eps']).fit(x, y)
    p_hub = max(float(hub.predict([[w - 1 + hz]])[0]), 0.0)
    ax.plot(x_ext, hub.predict([[0], [w - 1 + hz]]).flatten(),
            '--', color='green', linewidth=1.5, label=f'Huber={p_hub:.2f}')
except Exception:
    p_hub = p_ols

# Lasso
try:
    las = Lasso(alpha=PARAM['lasso_alpha']).fit(x, y)
    p_las = max(float(las.predict([[w - 1 + hz]])[0]), 0.0)
    ax.plot(x_ext, las.predict([[0], [w - 1 + hz]]).flatten(),
            '--', color='purple', linewidth=1.5, label=f'Lasso={p_las:.2f}')
except Exception:
    p_las = p_ols

# Mediana
p_med = max(float(np.median(y)), 0.0)
ax.axhline(p_med, color='orange', linestyle=':', linewidth=1.5,
           label=f'Mediana={p_med:.2f}')

# punto de predicción
for val, color in [(p_ols,'tomato'),(p_hub,'green'),(p_las,'purple'),(p_med,'orange')]:
    ax.scatter([t_pred], [val], color=color, s=80, zorder=6, marker='D')

ax.set_xticks(list(t_ult) + [t_pred])
ax.set_xticklabels(
    [str(p) for p in periodos[-w:]] + [f't+{hz}'],
    rotation=45, fontsize=7
)
ax.set_title(f'Últimos {v} meses + predicción t+{hz} — distintos modelos', fontsize=10)
ax.set_ylabel('tn')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle(f'Exploratorio — producto {pid} × cliente {cid}', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print(f"\nPredicciones para t+{hz}:")
print(f"  OLS:     {p_ols:.4f}")
print(f"  Huber:   {p_hub:.4f}")
print(f"  Lasso:   {p_las:.4f}")
print(f"  Mediana: {p_med:.4f}")

# Estadísticas descriptivas de la serie

In [ ]:
s = tn[tn > 0]
print(f"Producto {pid} × cliente {cid}")
print(f"  Períodos totales:     {len(tn)}")
print(f"  Períodos con venta:   {len(s)} ({100*len(s)/len(tn):.1f}%)")
print(f"  Períodos en cero:     {(tn==0).sum()}")
print()
print(f"  Media:                {tn.mean():.4f}")
print(f"  Mediana:              {np.median(tn):.4f}")
print(f"  Std:                  {tn.std():.4f}")
print(f"  CV (std/media):       {tn.std()/(tn.mean()+1e-9):.4f}")
print(f"  Min:                  {tn.min():.4f}")
print(f"  Max:                  {tn.max():.4f}")
print()
print(f"  Últimos {PARAM['ventana']} meses:")
ult = tn[-PARAM['ventana']:]
print(f"    media:    {ult.mean():.4f}")
print(f"    mediana:  {np.median(ult):.4f}")
print(f"    pendiente OLS: {float(LinearRegression().fit(np.arange(len(ult)).reshape(-1,1), ult).coef_[0]):.4f} tn/mes")

# detección de picos
umbral_pico = tn.mean() + 2 * tn.std()
picos = [(periodos[i], tn[i]) for i in range(len(tn)) if tn[i] > umbral_pico]
print(f"\n  Picos anómalos (> media+2σ = {umbral_pico:.2f}):")
if picos:
    for p, v in picos:
        print(f"    periodo {p}: {v:.4f}")
else:
    print("    ninguno")

# Cambiar producto o cliente

Modificá `PARAM` al principio y volvé a correr desde la celda de selección.

```python
PARAM['product_id']  = 12345   # ID específico
PARAM['customer_id'] = 67890
PARAM['ventana']     = 12      # más meses
PARAM['huber_eps']   = 1.0     # más robusto a outliers
```

O dejá `None` para que autoseleccione el menos importante.